# Imports

In [1]:
# Standard Library
import html
import itertools
import os
import re
import json

# Third-Party
import boto3
from botocore.config import Config
from dotenv import load_dotenv
from IPython.display import HTML, display
import pandas as pd
from tqdm import tqdm

# Local Application / Internal Modules
from bedrock import (
    BedrockConfig,
    extract_single_section,
)
from prompts import (
    AVAILABILITY_PROMPT,
    EDUCATION_PROMPT,
    JOB_SOFTSKILLS_PARSER_PROMPT,
    JOB_TOOLS_PARSER_PROMPT,
    MIN_EXPERIENCE_PROMPT,
)
from utils import (
    extract_and_move_advantages,
    omit_quoted_sentences,
    slice_job_sections,
    validate_education,
    validate_min_experience_years,
    validate_soft_skills,
    validate_hard_skills,
    detect_section_start_lines,
    truncate_unrelated_tail,
    display_job_comparison
)

# Environment Setup
load_dotenv()

True

# Configurations

In [2]:
# --- Data Configuration ---
START = 0
STOP = 1

# --- Core Project Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "eu-west-1")

# --- Initialize AWS Clients ---
boto_config = Config(
    retries={'max_attempts': 5, 'mode': 'standard'},
    max_pool_connections=50 
)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION, config=boto_config)

# --- LLM Configuration ---
MODEL_NAME = "google.gemma-3-4b-it"
MAX_GENERATION_TOKENS = 768

config = BedrockConfig(
    bedrock_runtime=bedrock_runtime,
    model_name=MODEL_NAME,
    max_generation_tokens=MAX_GENERATION_TOKENS,
)

# Load Dataset

In [3]:
#file_name = "job_posts"
file_name = "demo_jobs"

with open(f"../data/{file_name}.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Pattern to isolate each job block
pattern = r"(?im)^\s*job\s*\d+\s*$(.*?)(?=(?:^\s*job\s*\d+\s*$)|\Z)"
iterator = re.finditer(pattern, text, flags=re.DOTALL | re.MULTILINE)

# Pattern to capture metadata lines like "- Key: Value"
field_pattern = re.compile(r"^\s*-\s*([^:\n]+)\s*:\s*(.*)$", re.MULTILINE)


def parse_job_block(raw_block: str) -> dict:
    data = {}
    lines = raw_block.strip().splitlines()
    text_content_lines = []
    collecting_text_body = False

    for line in lines:
        if collecting_text_body:
            text_content_lines.append(line)
            continue

        match = field_pattern.match(line)
        if match:
            key = match.group(1).strip()
            val = match.group(2).strip()

            if key.lower() == "text":
                # If there's inline text (e.g. "- Text: some description")
                if val:
                    text_content_lines.append(val)
                collecting_text_body = True
            else:
                data[key] = val
        else:
            # Fallback in case there are lines before or outside the headers
            if collecting_text_body or line.strip():
                text_content_lines.append(line)

    data["Text"] = "\n".join(text_content_lines).strip()
    return data


# Slice and parse into structured rows
parsed_jobs = [
    parse_job_block(m.group(1))
    for m in itertools.islice(iterator, START, STOP)
]

df = pd.DataFrame(parsed_jobs)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head()

,job_title,employment_type,location,company_name,text
0,Backend Engineer(Data),Full Time,Tel Aviv-Yafo,DoiT,The Opportunity\n\nAttribute technology and se...


# Preprocess and Split Sections

In [5]:
# Step 1: Normalize inline advantages
df["text"] = df["text"].apply(extract_and_move_advantages)

# Step 2: Detect section boundaries
df['markers'] = df['text'].apply(detect_section_start_lines)

# Step 3: Slice sections and assign directly into DataFrame columns
tqdm.pandas(desc="Slicing Sections")
sliced_sections = [
    slice_job_sections(text, markers)
    for text, markers in zip(df["text"], df["markers"])
]

# Unpack sliced dictionary records into columns in one step
sections_df = pd.DataFrame(sliced_sections, index=df.index)
df[["responsibilities", "must_have", "nice_to_have"]] = sections_df[
    ["responsibilities", "must_have", "nice_to_have"]
]

def clean_final_section(row: pd.Series) -> pd.Series:

    row = row.copy()
    
    # Read values safely (handles None, NaN, float, or string)
    nice_to_have = str(row.get("nice_to_have", "") or "").strip()
    must_have = str(row.get("must_have", "") or "").strip()
    
    # Check if nice_to_have has real content
    if nice_to_have and nice_to_have.upper() not in ["NONE", "NAN"]:
        row["nice_to_have"] = truncate_unrelated_tail(nice_to_have)
    # Fallback to must_have
    elif must_have and must_have.upper() not in ["NONE", "NAN"]:
        row["must_have"] = truncate_unrelated_tail(must_have)
        
    return row

df = df.apply(clean_final_section, axis=1)

df.head()

,job_title,employment_type,location,company_name,text,markers,responsibilities,must_have,nice_to_have
0,Backend Engineer(Data),Full Time,Tel Aviv-Yafo,DoiT,The Opportunity\n\nAttribute technology and se...,"{'responsibilities_start': 'Responsibilities',...",Responsibilities\n\nImplementing features. Fro...,Qualifications\n\n3+ years of hands-on experie...,Bonus Points\n\nExperience with C/C#\nFamiliar...
1,Senior Data Engineer,None,None,AcmeCorp,About AcmeCorp:\nWe are hiring a Senior Data E...,{'responsibilities_start': 'Responsibilities:'...,Responsibilities:\n- Build resilient ETL pipel...,Requirements:\n- 5+ years of Python software d...,Advantages:\n- Hands-on experience with AWS Be...


# Extract Info From Job Post

In [6]:
# ==============================================================================
# Pipeline Helpers
# ==============================================================================

def merge_qualifications(
    base_list: list[str], new_list: list[str]
) -> list[str]:
    """Appends newly extracted items to the base list."""
    base = base_list if isinstance(base_list, list) else []
    added = new_list if isinstance(new_list, list) else []

    valid_added = [
        item
        for item in added
        if item.strip() and "Soft Skills Sentence: None" not in item
    ]
    return base + valid_added

def run_extraction_pipeline(
    df: pd.DataFrame, source_col: str, steps: list
) -> tuple[list[list[str]], list[dict[str, str]]]:
    """Iteratively extracts criteria, merges results, and tracks working_text evolution.

    Returns:
        (results, text_history)
        - results: list of extracted criteria per row
        - text_history: list of dicts per row mapping step name to working text
    """
    n_rows = len(df)
    results = [[] for _ in range(n_rows)]

    # Initialize per-row history recording the pristine starting text
    history = [{"0. Initial": str(t or "")} for t in df[source_col]]
    working_text = df[source_col].copy()

    for step in steps:
        # Unpack step with optional 4th and 5th arguments
        desc = step[0]
        prompt = step[1]
        sections = step[2]
        validation_fn = step[3] if len(step) > 3 else None
        user_template = step[4] if len(step) > 4 else None

        tqdm.pandas(desc=desc)

        # 1. Standard section extraction
        extracted = working_text.progress_apply(
            lambda t: extract_single_section(
                text=t, 
                system_prompt=prompt, 
                cfg=config,
                allowed_sections=sections, 
                user_template=user_template
            )
        )
        
        # 2. Apply validation function if provided
        if validation_fn:
            extracted = pd.Series(
                [validation_fn(ext, src) for ext, src in zip(extracted, working_text)],
                index=extracted.index
            )

        # 3. Omit extracted sentences from remaining text
        updated_text = [
            omit_quoted_sentences(t, e)
            for t, e in zip(working_text, extracted)
        ]
        working_text = pd.Series(updated_text, index=df.index)

        # 4. Accumulate extracted results
        results = [
            merge_qualifications(b, e) for b, e in zip(results, extracted)
        ]

        # 5. Save snapshot of working_text after this pass for every row
        for i, val in enumerate(working_text):
            history[i][desc] = str(val or "")

    return results, history

# ==============================================================================
# Pipeline Execution
# ==============================================================================

# Format: (Description, System Prompt, Allowed Sections, Validation Function, User Template)
MANDATORY_STEPS = [
    ("Pass 1: Education",     EDUCATION_PROMPT,             ["EDUCATION"],                    validate_education,               None),
    ("Pass 2: Availability",  AVAILABILITY_PROMPT,          ["AVAILABILITY"],                 None,                             None),
    ("Pass 3: Min Exp",       MIN_EXPERIENCE_PROMPT,        ["REQUIRED YEARS OF EXPERIENCE"], validate_min_experience_years,    None),
    ("Pass 4: Soft Skills",   JOB_SOFTSKILLS_PARSER_PROMPT, ["SOFT SKILLS"],                  validate_soft_skills,             None),
    ("Pass 5: Hard Skills",   JOB_TOOLS_PARSER_PROMPT,      None,                             None,             None),
]

ADVANTAGE_STEPS = [
    ("Pass 1: Education",             EDUCATION_PROMPT,             ["EDUCATION"],            validate_education,               None),
    ("Pass 2: Advantage Soft Skills", JOB_SOFTSKILLS_PARSER_PROMPT, ["SOFT SKILLS"],          validate_soft_skills,             None),
    ("Pass 3: Advantage Hard Skills", JOB_TOOLS_PARSER_PROMPT,      None,                     validate_hard_skills,             None)
]

# Run pipelines and assign everything directly to a single column per category
df["mandatory"], df["mandatory_history"] = run_extraction_pipeline(df, "must_have", MANDATORY_STEPS)
df["advantages"], df["advantages_history"] = run_extraction_pipeline(df, "nice_to_have", ADVANTAGE_STEPS)

Pass 1: Education:   0%|          | 0/2 [00:00<?, ?it/s]

Pass 3: Advantage Hard Skills: 100%|██████████| 2/2 [00:01<00:00,  1.25it/s]


# Inspect Extraction

In [7]:
display_job_comparison(
    df, idx=1, 
    text_col="text", 
    mand_col="mandatory", 
    adv_col="advantages",
)

In [7]:
def display_text_history(
    df: pd.DataFrame,
    idx: int,
    history_col: str,
    render_html: bool = True,
    font_size: int = 16,
):
    """Prints or renders all progressive versions of working_text for a single row."""
    if idx not in df.index:
        print(f"Index {idx} not found in DataFrame.")
        return

    versions = df.loc[idx, history_col]
    if not isinstance(versions, dict):
        print(f"No valid history dictionary found at index {idx}.")
        return

    # Plain text console output
    if not render_html:
        print(
            f"\n{'='*80}\nWORKING TEXT HISTORY: Index {idx} [{history_col}]\n{'='*80}"
        )
        for step_name, text in versions.items():
            print(f"\n--- [{step_name}] ---")
            print(f"Char length: {len(text)}")
            print(f"{'-'*40}\n{text if text.strip() else '<EMPTY>'}")
        return

    # Notebook HTML rendering
    cards = []
    for step_name, text in versions.items():
        escaped_text = (
            html.escape(text.strip())
            if text.strip()
            else "<span style='color: #777;'>[EMPTY TEXT]</span>"
        )
        cards.append(f"""
        <div style="margin-bottom: 18px; border: 1px solid #3c3c3c; border-radius: 8px; background: #1e1e1e; overflow: hidden;">
            <div style="background: #2d2d2d; padding: 10px 16px; display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #3c3c3c;">
                <span style="font-weight: 600; font-size: 15px; color: #4ec9b0; font-family: monospace;">{html.escape(step_name)}</span>
                <span style="font-size: 13px; color: #aaa;">Length: {len(text)} chars</span>
            </div>
            <pre style="margin: 0; padding: 18px; white-space: pre-wrap; word-break: break-word; font-size: {font_size}px; font-family: 'Consolas', 'Fira Code', 'Courier New', monospace; color: #e6e6e6; line-height: 1.7;">{escaped_text}</pre>
        </div>
        """)

    container = f"""
    <div style="max-width: 1200px; padding: 20px; background: #141414; border-radius: 10px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <h3 style="margin: 0 0 18px 0; color: #fff; font-size: 20px;">Text Evolution for Row {idx} ({history_col})</h3>
        {''.join(cards)}
    </div>
    """
    display(HTML(container))

display_text_history(df, idx=2, history_col="advantages_history")

In [ ]:
KV_REGEX = re.compile(r"^([^:]+):\s*(.*)$")
MIN_EXP_KEY_REGEX = re.compile(r"^Min Experience Years(?:\s+\d+)?$", re.IGNORECASE)

ARRAY_FIELDS = (
    "Hard Skills",
    "Advantage - Hard Skills",
    "Min Experience Years",
    "Soft Skills",
    "Advantage - Soft Skills",
)

# Clean up dataframe columns if they exist
if "responsibilities" in df.columns:
    df["responsibilities"] = df["responsibilities"].fillna("")

def _is_empty_token(val) -> bool:
    if val is None:
        return True
    s = str(val).strip()
    return s.upper() in ("", "NONE", "NAN") or s.lower().startswith("none [quote:")

def ensure_string_list(val) -> list[str]:
    """Normalizes None, 'None', a single string, or a list into a clean list[str]."""
    if _is_empty_token(val):
        return []
    if isinstance(val, (list, tuple)):
        return [str(v).strip() for v in val if not _is_empty_token(v)]
    return [str(val).strip()]

def parse_list_to_dict(items: list | None) -> dict:
    if not isinstance(items, (list, tuple)):
        return {}

    result = {}
    for item in items:
        item = str(item).strip()
        match = KV_REGEX.match(item)
        if not match:
            continue

        raw_key = match.group(1).strip()
        val = match.group(2).strip()

        # Collapse "Min Experience Years 1", "Min Experience Years 2", etc. into "Min Experience Years"
        if MIN_EXP_KEY_REGEX.match(raw_key):
            key = "Min Experience Years"
            if not _is_empty_token(val):
                result.setdefault(key, []).append(val)
            else:
                result.setdefault(key, [])
            continue

        if raw_key in result:
            if isinstance(result[raw_key], list):
                result[raw_key].append(val)
            else:
                result[raw_key] = [result[raw_key], val]
        else:
            result[raw_key] = val

    return result

def row_to_json_record(row: dict) -> dict:
    record = row.copy()

    record.pop("mandatory_history", None)
    record.pop("advantages_history", None)
    record.pop("markers", None)
    record.pop("must_have", None)
    record.pop("nice_to_have", None)

    # Flatten mandatory fields
    record.update(parse_list_to_dict(record.pop("mandatory", [])))

    # Flatten advantages fields (prefixed if keys collide)
    adv_dict = parse_list_to_dict(record.pop("advantages", []))
    for k, v in adv_dict.items():
        key_name = f"Advantage - {k}" if k in record else k
        record[key_name] = v

    # Guarantee array fields are always lists
    for field in ARRAY_FIELDS:
        record[field] = ensure_string_list(record.get(field))

    return record 

records = [row_to_json_record(row) for row in df.to_dict(orient="records")]

with open("jobs_extracted.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)